In [ ]:
!pip install -q langchain langchain-text-splitters langchain-chroma langchain-huggingface pypdf sentence-transformers
!pip install -q transformers accelerate bitsandbytes

In [ ]:
!pip install -q langchain-community

In [ ]:
research_paper_filename = "research.pdf"

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

pdf = PyPDFLoader("research.pdf")
docs = pdf.load()

print(f"Loaded {len(docs)} pages")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

embed = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

db = Chroma.from_documents(
    documents=chunks,
    embedding=embed,
    persist_directory="./chroma_db"
)

In [ ]:
!pip install -q langchain-groq

import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "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"

model = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.1
)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

temp = """You are an assistant that answers questions about a research paper.

Use only the given context to answer the question. If the answer is not present, say "I don't see that information in the research paper."

Context:
{context}

Question:
{question}

Answer:"""

prompt = PromptTemplate(
    template=temp,
    input_variables=["context", "question"]
)

def get_text(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = db.as_retriever(search_kwargs={"k": 10})

chain = (
    {
        "context": retriever | get_text,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
q = "What are the main applications of Artificial Intelligence discussed in this research paper?"

print("Question:", q)
print("\nSearching research paper...\n")

docs = retriever.invoke(q)
print(f"Found {len(docs)} relevant sections\n")

ans = chain.invoke(q)

print("Answer:\n")
print(ans)

print("\n" + "=" * 50)
print("Sources Used:\n")

for d in docs:
    text = d.page_content.replace("/envel⌢", "")
    print(text)
    print("-" * 50)